In [12]:
# Please execute/shift-return this cell everytime you run the notebook.  Don't edit it. 
%load_ext autoreload
%autoreload 2
from notebook import * 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Case study: matrix multiplications

GEMM that computes C = A $\times$ B is the core of many AI/ML applications. The most naive implementation of GEMM takes $O(n^3)$. Assume it takes 1 second to perform GEMM on 1,024$\times$1,024$\times$1,024 matrices. How much time do you expect it would take for 2,048$\times$2,048$\times$2,048 matrices?

In [2]:
render_code("matrix_mul/mm.c", show=["//START","//END"])

// matrix_mul/mm.c:54-75 (22 lines)
//START
void mm(double **a, double **b, double **c, uint64_t M, uint64_t N, uint64_t K)
{
  uint64_t i,j,k;
  for(i = 0; i < M; i++)
  {
    for(j = 0; j < K; j++)
    {
      for(k = 0; k < N; k++)
      {
        c[i][j] += a[i][k]*b[k][j];
        #ifdef DUMP
          fprintf(stderr, "a[%ld][%ld], %p\n",i,k, &a[i][k]);
          fprintf(stderr, "b[%ld][%ld], %p\n",k,j, &b[k][j]);
          fprintf(stderr, "c[%ld][%ld], %p\n",i,j, &c[i][j]);
        #endif
      }
    }
  }
  return;
}
//END

In [4]:
! cd matrix_mul; make clean; make mm

rm -f blockmm mm blockmm_transpose cachegrind.* mm_dump rect_blockmm_trans blockmm_transpose_reg blockmm_reg
gcc -DHAVE_LINUX_PERF_EVENT_H -O3 mm.c perfstats.c -o mm 


In [5]:
! cd matrix_mul; echo "IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > mm.csv
! ./matrix_mul/mm 512 >> ./matrix_mul/mm.csv ;./matrix_mul/mm 1024 >> ./matrix_mul/mm.csv ; ./matrix_mul/mm 2048 >> ./matrix_mul/mm.csv
#! cs203 job memory "./matrix_mul/mm 1024 >> ./matrix_mul/mm.csv ; ./matrix_mul/mm 2048 >> ./matrix_mul/mm.csv"

234410496.000000,1406510080.000000,10521102336.000000,

In [5]:
display_df_mono(render_csv("matrix_mul/mm.csv"))

,index,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,512,1077385407,733715945,0.681015,0.179304,0.131558,0.250675,134788138,537700725
1,1024,8619376766,10109518735,1.172883,0.179041,1.810018,0.233160,1003459254,4303731056
2,2048,68985106847,121737841162,1.764697,0.179266,21.823417,0.306432,10555476762,34446373893


WOW! Compuational complexty breaks again! The GEMM performance go wild because of cache misses!

What kind of misses are we seeing?

In [6]:
! make -C matrix_mul mm_dump; ./matrix_mul/mm_dump 256 >& mm_dump_address.csv

make: Entering directory '/nfshome/htseng/courses/CS203/demo/memory/matrix_mul'
gcc -DHAVE_LINUX_PERF_EVENT_H -DDUMP -O3 mm.c perfstats.c -o mm_dump 
make: Leaving directory '/nfshome/htseng/courses/CS203/demo/memory/matrix_mul'


In [7]:
! echo "element,address" > mm_dump_addresses_digest.csv 
! head -n 101 mm_dump_address.csv | grep "b\[" >> mm_dump_addresses_digest.csv
df = pd.read_csv("mm_dump_addresses_digest.csv",skipfooter=1,engine='python')
df["address"] = df["address"].str.replace('0x','')
df["address"]=df[["address"]].apply(lambda x: x.astype(str).map(lambda x: int(x, base=16)))
# only show the first N addresses 
#N = 32
#df2 = df2.iloc[:N]
C = 49152
B = 64
A = 12
offset_bits = int(math.log2(B))
S = int(C/(B*A))
index_bits = int(math.log2(S))
df["tag"]=(df["address"].apply(lambda x: x >> (offset_bits+index_bits)))
df["tag"] = df["tag"].apply(lambda x: hex(x))
df["index"] = df["address"].apply(lambda x: hex((x>>offset_bits)%S))
df["address"] = df["address"].apply(lambda x: hex(x))
display_df_mono(df)

,element,address,tag,index
0,b[0][0],0x7b7c52eff000,0x7b7c52eff,0x0
1,b[1][0],0x7b7c52eff800,0x7b7c52eff,0x20
2,b[2][0],0x7b7c52f00000,0x7b7c52f00,0x0
3,b[3][0],0x7b7c52f00800,0x7b7c52f00,0x20
4,b[4][0],0x7b7c52f01000,0x7b7c52f01,0x0
5,b[5][0],0x7b7c52f01800,0x7b7c52f01,0x20
6,b[6][0],0x7b7c52f02000,0x7b7c52f02,0x0
7,b[7][0],0x7b7c52f02800,0x7b7c52f02,0x20
8,b[8][0],0x7b7c52f03000,0x7b7c52f03,0x0
9,b[9][0],0x7b7c52f03800,0x7b7c52f03,0x20


### Matrix tiling algorithm

Let's try to partition GEMM into smaller tiles!

In [6]:
render_code("matrix_mul/blockmm.c", show=["//START","//END"])

// matrix_mul/blockmm.c:59-77 (19 lines)
//START
void blockmm(double **a, double **b, double **c, uint64_t M, uint64_t N, uint64_t K)
{
  uint64_t i,j,k, ii, jj, kk;
  for(i = 0; i < M; i+=tile_size)
  {
    for(j = 0; j < K; j+=tile_size)
    {
      for(k = 0; k < N; k+=tile_size)
      {        
          for(ii = i; ii < i+tile_size; ii++)
            for(jj = j; jj < j+tile_size; jj++)
              for(kk = k; kk < k+tile_size; kk++)
                c[ii][jj] += a[ii][kk]*b[kk][jj];
      }
    }
  }  
}
//END

In [7]:
! cd matrix_mul/; make clean blockmm

rm -f blockmm mm blockmm_transpose cachegrind.* mm_dump rect_blockmm_trans blockmm_transpose_reg blockmm_reg
gcc -O4 -DHAVE_LINUX_PERF_EVENT_H blockmm.c perfstats.c -o blockmm 
blockmm.c: In function ‘main’:
blockmm.c:48:16: warning: format ‘%lu’ expects argument of type ‘long unsigned int’, but argument 3 has type ‘int’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wformat=-Wformat=]8;;]
   48 |   printf("%d,%lu,",ARRAY_SIZE,tile_size);
      |              ~~^              ~~~~~~~~~
      |                |              |
      |                |              int
      |                long unsigned int
      |              %u


## Try with tile size == 32

In [25]:
! cd matrix_mul; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm.csv
! ./matrix_mul/blockmm 512 32 >> ./matrix_mul/blockmm.csv ;./matrix_mul/blockmm 1024 32 >> ./matrix_mul/blockmm.csv ; ./matrix_mul/blockmm 2048 32 >> ./matrix_mul/blockmm.csv; ./matrix_mul/blockmm 4096 32 >> ./matrix_mul/blockmm.csv

In [26]:
display_df_mono(render_csv("matrix_mul/mm.csv"))
display_df_mono(render_csv("matrix_mul/blockmm.csv"))

,index,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,512,1061982841,705546177,0.664367,0.196142,0.138387,0.251255,133173447,530033241
1,1024,8514486659,10689402335,1.255437,0.195400,2.088713,0.233153,991229199,4251417195


,index,size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,512,32,1005794817,356370501,0.354317,0.230895,0.082284,0.213914,105909936,495106058
1,1,1024,32,8940674486,3229675980,0.361234,0.193332,0.624401,0.211290,929881888,4400969812
2,2,2048,32,71536204837,28167079933,0.393746,0.193309,5.444939,0.216990,7640613155,35211822273
3,3,4096,32,572225186023,235886586814,0.412227,0.193321,45.601793,0.222570,62689586507,281662146815


## Try with tile size == 8

In [27]:
! cd matrix_mul; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm.csv
! ./matrix_mul/blockmm 512 8 >> ./matrix_mul/blockmm.csv ;./matrix_mul/blockmm 1024 8 >> ./matrix_mul/blockmm.csv ; ./matrix_mul/blockmm 2048 8 >> ./matrix_mul/blockmm.csv; ./matrix_mul/blockmm 4096 8 >> ./matrix_mul/blockmm.csv

In [28]:
display_df_mono(render_csv("matrix_mul/mm.csv"))
display_df_mono(render_csv("matrix_mul/blockmm.csv"))

,index,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,512,1061982841,705546177,0.664367,0.196142,0.138387,0.251255,133173447,530033241
1,1024,8514486659,10689402335,1.255437,0.195400,2.088713,0.233153,991229199,4251417195


,index,size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,512,8,1037973426,218646239,0.210647,0.274201,0.059953,0.008980,4388021,488640541
1,1,1024,8,9959664173,2169682389,0.217847,0.199811,0.433526,0.012892,60447172,4688638501
2,2,2048,8,81044513879,23949840822,0.295515,0.193237,4.628006,0.010680,407441799,38151468383
3,3,4096,8,648374819796,218947626303,0.337687,0.193288,42.319855,0.012642,3858359715,305213183566


In [11]:
! ./matrix_mul/blockmm 2048 4 >> ./matrix_mul/blockmm.csv
! ./matrix_mul/blockmm 2048 16 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 32 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 64 >> ./matrix_mul/blockmm.csv
! ./matrix_mul/blockmm 2048 128 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 256 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 512 >> ./matrix_mul/blockmm.csv 
display_df_mono(render_csv("matrix_mul/blockmm.csv"))

,index,size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,512,8,1147298416,243190881,0.211968,0.238200,0.057928,0.009093,4911340,540104644
1,1,1024,8,9670211410,2044111131,0.211382,0.196899,0.402484,0.012531,57044013,4552227076
2,2,2048,8,80851755795,18305093979,0.226403,0.179967,3.294311,0.010248,390047768,38060634362
3,3,4096,8,648409985898,194455500601,0.299896,0.179027,34.812841,0.012341,3766813265,305226020159
4,4,2048,4,97764512409,24052250240,0.246022,0.179020,4.305843,0.015264,666144284,43640871641
5,5,2048,16,74471875927,19105878386,0.256552,0.178996,3.419875,0.071624,2585932924,36104492109
6,6,2048,32,71542023381,27622340203,0.386100,0.178988,4.944074,0.217019,7642037440,35213614814
7,7,2048,64,70147541762,32507116711,0.463411,0.179142,5.823387,0.242676,8443529551,34793370875
8,8,2048,128,69466685540,34560810295,0.497516,0.179006,6.186601,0.245787,8501481880,34588840472
9,9,2048,256,69126122044,35104121492,0.507827,0.179013,6.284101,0.247019,8518696402,34485987806


In [73]:
render_code("matrix_mul/blockmm_reg.c", show=["//START","//END"])

// matrix_mul/blockmm_reg.c:59-82 (24 lines)
//START
void blockmm(double **a, double **b, double **c, uint64_t M, uint64_t N, uint64_t K)
{
  uint64_t i,j,k, ii, jj, kk;
    double result = 0;
  for(i = 0; i < M; i+=tile_size)
  {
    for(j = 0; j < K; j+=tile_size)
    {
      for(k = 0; k < N; k+=tile_size)
      {        
          for(ii = i; ii < i+tile_size; ii++)
            for(jj = j; jj < j+tile_size; jj++)
                {
                result = 0;
                for(kk = k; kk < k+tile_size; kk++)
                    result += a[ii][kk]*b[kk][jj];
                c[ii][jj] += result;
          }
      }
    }
  }  
}
//END

In [72]:
! cd matrix_mul; make blockmm_reg; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm_reg.csv
! ./matrix_mul/blockmm_reg 2048 4 >> ./matrix_mul/blockmm_reg.csv
! ./matrix_mul/blockmm_reg 2048 8 >> ./matrix_mul/blockmm_reg.csv
! ./matrix_mul/blockmm_reg 2048 16 >> ./matrix_mul/blockmm_reg.csv 
! ./matrix_mul/blockmm_reg 2048 32 >> ./matrix_mul/blockmm_reg.csv 
! ./matrix_mul/blockmm_reg 2048 64 >> ./matrix_mul/blockmm_reg.csv
! ./matrix_mul/blockmm_reg 2048 128 >> ./matrix_mul/blockmm_reg.csv 
! ./matrix_mul/blockmm_reg 2048 256 >> ./matrix_mul/blockmm_reg.csv 
! ./matrix_mul/blockmm_reg 2048 512 >> ./matrix_mul/blockmm_reg.csv 
display_df_mono(render_csv("matrix_mul/blockmm_reg.csv"))

make: 'blockmm_reg' is up to date.


,index,size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,2048,4,106201212076,27845781930,0.262198,0.193551,5.389566,0.018315,772133133,42157533026
1,1,2048,8,78810598126,18401319177,0.233488,0.193465,3.560020,0.012768,393063134,30785103090
2,2,2048,16,66922355202,15623019363,0.233450,0.193700,3.026183,0.099929,2591916360,25937601095
3,3,2048,32,61315357221,18353349348,0.299327,0.193552,3.552330,0.296214,7012269806,23672978725
4,4,2048,64,58581226044,20759230914,0.354367,0.193442,4.015697,0.378735,8549539168,22573950649
5,5,2048,128,57224812063,20347113839,0.355565,0.193357,3.934262,0.393992,8679687393,22030110590
6,6,2048,256,56566693722,28292963940,0.500170,0.193405,5.471995,0.397981,8662481717,21766088957
7,7,2048,512,56260252814,42224816816,0.750527,0.193520,8.171333,0.399518,8646624419,21642618429


In [59]:
render_code("matrix_mul/blockmm_transpose.c", show=["//START","//END"])

// matrix_mul/blockmm_transpose.c:62-80 (19 lines)
//START
void blockmm_transpose(double **a, double **b, double **c, uint64_t M, uint64_t N, uint64_t K)
{
  int i,j,k, ii, jj, kk;
  for(i = 0; i < M; i+=tile_size)
  {
    for(j = 0; j < K; j+=tile_size)
    {
      for(k = 0; k < N; k+=tile_size)
      {        
          for(ii = i; ii < i+tile_size; ii++)
            for(jj = j; jj < j+tile_size; jj++)
              for(kk = k; kk < k+tile_size; kk++)
                c[ii][jj] += a[ii][kk]*b[jj][kk];
      }
    }
  }  
}
//END

### Matrix transpose

In [13]:
! cd matrix_mul; rm blockmm_transpose; make blockmm_transpose; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm_transpose.csv
! ./matrix_mul/blockmm_transpose 512 8 >> ./matrix_mul/blockmm_transpose.csv ;./matrix_mul/blockmm_transpose 1024 8 >> ./matrix_mul/blockmm_transpose.csv ; ./matrix_mul/blockmm_transpose 2048 8 >> ./matrix_mul/blockmm_transpose.csv; ./matrix_mul/blockmm_transpose 4096 8 >> ./matrix_mul/blockmm_transpose.csv

rm: cannot remove 'blockmm_transpose': No such file or directory
gcc -O4 -DHAVE_LINUX_PERF_EVENT_H blockmm_transpose.c perfstats.c -o blockmm_transpose
234410496.000000,1406510080.000000,10521102336.000000,48070299648.000000,

In [14]:
! ./matrix_mul/blockmm_transpose 2048 8 >> ./matrix_mul/blockmm_transpose.csv 
! ./matrix_mul/blockmm_transpose 2048 16 >> ./matrix_mul/blockmm_transpose.csv 
! ./matrix_mul/blockmm_transpose 2048 32 >> ./matrix_mul/blockmm_transpose.csv 
! ./matrix_mul/blockmm_transpose 2048 64 >> ./matrix_mul/blockmm_transpose.csv
! ./matrix_mul/blockmm_transpose 2048 128 >> ./matrix_mul/blockmm_transpose.csv
! ./matrix_mul/blockmm_transpose 2048 256 >> ./matrix_mul/blockmm_transpose.csv

10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,

In [15]:
display_df_mono(render_csv("matrix_mul/blockmm_transpose.csv"))

,index,size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,512,8,1002238747,231622728,0.231105,0.220229,0.051010,0.002593,1062791,409859001
1,1,1024,8,8298791736,1913293069,0.230551,0.204950,0.392130,0.002053,6967116,3393857125
2,2,2048,8,70362448430,16492706173,0.234396,0.191508,3.158491,0.005796,166771629,28775095210
3,3,4096,8,562464311715,132646927120,0.235832,0.179436,23.801608,0.002280,524539372,230027483826
4,4,2048,8,70387748918,16498212768,0.234390,0.179151,2.955666,0.002527,72744027,28786997355
5,5,2048,16,64859570251,13814086335,0.212985,0.179744,2.483004,0.022735,615407468,27068179192
6,6,2048,32,62422857231,14941697042,0.239363,0.179681,2.684744,0.041281,1088882210,26377413168
7,7,2048,64,61309414210,13767663175,0.224560,0.179035,2.464889,0.024152,630028864,26085495645
8,8,2048,128,60764372416,17188982377,0.282879,0.179077,3.078151,0.018634,483481113,25945589778
9,9,2048,256,60492948668,17291693419,0.285846,0.179009,3.095373,0.011701,302778053,25876707458


In [16]:
display_df_mono(render_csv("matrix_mul/blockmm.csv"))

,index,size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,512,8,1147298416,243190881,0.211968,0.238200,0.057928,0.009093,4911340,540104644
1,1,1024,8,9670211410,2044111131,0.211382,0.196899,0.402484,0.012531,57044013,4552227076
2,2,2048,8,80851755795,18305093979,0.226403,0.179967,3.294311,0.010248,390047768,38060634362
3,3,4096,8,648409985898,194455500601,0.299896,0.179027,34.812841,0.012341,3766813265,305226020159
4,4,2048,4,97764512409,24052250240,0.246022,0.179020,4.305843,0.015264,666144284,43640871641
5,5,2048,16,74471875927,19105878386,0.256552,0.178996,3.419875,0.071624,2585932924,36104492109
6,6,2048,32,71542023381,27622340203,0.386100,0.178988,4.944074,0.217019,7642037440,35213614814
7,7,2048,64,70147541762,32507116711,0.463411,0.179142,5.823387,0.242676,8443529551,34793370875
8,8,2048,128,69466685540,34560810295,0.497516,0.179006,6.186601,0.245787,8501481880,34588840472
9,9,2048,256,69126122044,35104121492,0.507827,0.179013,6.284101,0.247019,8518696402,34485987806


In [40]:
render_code("matrix_mul/blockmm_transpose_reg.c", show=["//START","//END"])

// matrix_mul/blockmm_transpose_reg.c:62-85 (24 lines)
//START
void blockmm_transpose(double **a, double **b, double **c, uint64_t M, uint64_t N, uint64_t K)
{
  int i,j,k, ii, jj, kk;
  double result = 0;
  for(i = 0; i < M; i+=tile_size)
  {
    for(j = 0; j < K; j+=tile_size)
    {
      for(k = 0; k < N; k+=tile_size)
      {        
          for(ii = i; ii < i+tile_size; ii++)
            for(jj = j; jj < j+tile_size; jj++)
            {
              result = 0;
              for(kk = k; kk < k+tile_size; kk++)
                result += a[ii][kk]*b[jj][kk];
              c[ii][jj] += result;
            }
      }
    }
  }  
}
//END

In [77]:
! cd matrix_mul; rm blockmm_transpose_reg; make blockmm_transpose_reg; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm_transpose_reg.csv 
! ./matrix_mul/blockmm_transpose_reg 2048 8 >> ./matrix_mul/blockmm_transpose_reg.csv 
! ./matrix_mul/blockmm_transpose_reg 2048 16 >> ./matrix_mul/blockmm_transpose_reg.csv 
! ./matrix_mul/blockmm_transpose_reg 2048 32 >> ./matrix_mul/blockmm_transpose_reg.csv 
! ./matrix_mul/blockmm_transpose_reg 2048 64 >> ./matrix_mul/blockmm_transpose_reg.csv
! ./matrix_mul/blockmm_transpose_reg 2048 128 >> ./matrix_mul/blockmm_transpose_reg.csv
! ./matrix_mul/blockmm_transpose_reg 2048 256 >> ./matrix_mul/blockmm_transpose_reg.csv

gcc -O4 -DHAVE_LINUX_PERF_EVENT_H blockmm_transpose_reg.c perfstats.c -o blockmm_transpose_reg
10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,

In [78]:
display_df_mono(render_csv("matrix_mul/blockmm_transpose_reg.csv"))

,index,size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,2048,8,60786299996,16265684823,0.267588,0.193655,3.149933,0.005718,90877802,15891907875
1,1,2048,16,49266996455,10288693657,0.208835,0.193558,1.991454,0.066974,804574967,12013255375
2,2,2048,32,43910010418,9894902746,0.225345,0.193504,1.914704,0.097494,1001490583,10272355840
3,3,2048,64,41303007641,10408599378,0.252006,0.193539,2.014475,0.074812,706106135,9438441181
4,4,2048,128,40016350092,11773600161,0.294220,0.193277,2.275566,0.051478,464828700,9029737941
5,5,2048,256,39380217413,13725019219,0.348526,0.193216,2.651894,0.021154,186754907,8828553246


In [69]:
render_code("matrix_mul/rect_blockmm_trans.c", show=["//START","//END"])

// matrix_mul/rect_blockmm_trans.c:73-96 (24 lines)
//START
void blockmm_transpose(double **a, double **b, double **c, uint64_t M, uint64_t N, uint64_t K)
{
  int i,j,k, ii, jj, kk;
  double result=0;
  for(i = 0; i < M; i+=tile_size_y)
  {
    for(j = 0; j < K; j+=tile_size_y)
    {
      for(k = 0; k < N; k+=tile_size_x)
      {
          for(ii = i; ii < i+tile_size_y; ii++)
              for(jj = j; jj < j+tile_size_y; jj++)
              {
                      result = 0;
                      for(kk = k; kk < k+tile_size_x ; kk++)
                          result += a[ii][kk]*b[jj][kk];
                      c[ii][jj] += result;
              }
      }
    }
  }  
}
//END

In [21]:
! cd matrix_mul; make rect_blockmm_trans; echo "size,tile_size_x,tile_size_y,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > rect_blockmm_trans.csv
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 8 8 >> ./matrix_mul/rect_blockmm_trans.csv 
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 8 16 >> ./matrix_mul/rect_blockmm_trans.csv 
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 16 8 >> ./matrix_mul/rect_blockmm_trans.csv
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 16 16 >> ./matrix_mul/rect_blockmm_trans.csv
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 32 8 >> ./matrix_mul/rect_blockmm_trans.csv 
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 32 16 >> ./matrix_mul/rect_blockmm_trans.csv 
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 64 8 >> ./matrix_mul/rect_blockmm_trans.csv
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 128 8 >> ./matrix_mul/rect_blockmm_trans.csv
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 256 8 >> ./matrix_mul/rect_blockmm_trans.csv
display_df_mono(render_csv("matrix_mul/rect_blockmm_trans.csv"))

gcc -O4 -DHAVE_LINUX_PERF_EVENT_H rect_blockmm_trans.c perfstats.c -o rect_blockmm_trans
10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,

,index,size,tile_size_x,tile_size_y,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,2048,8,8,60695062784,11689771085,0.192598,0.179022,2.092725,0.003680,58265221,15831612197
1,1,2048,8,16,59756448082,12532505081,0.209726,0.178994,2.243240,0.021419,329311143,15374651919
2,2,2048,16,8,49688800765,11470869620,0.230854,0.178995,2.053230,0.005624,68702899,12215974685
3,3,2048,16,16,49214866100,10038211829,0.203967,0.178982,1.796656,0.067128,804582256,11985747121
4,4,2048,32,8,44181716556,9419448705,0.213198,0.179072,1.686763,0.006081,63285810,10406706480
5,5,2048,32,16,43946350104,9800847644,0.223018,0.178981,1.754168,0.066056,679858043,10292123874
6,6,2048,64,8,41430771666,9723376378,0.234690,0.178995,1.740439,0.005216,49565585,9503023400
7,7,2048,128,8,40059251989,11698172180,0.292022,0.179010,2.094095,0.006316,57174965,9052631804
8,8,2048,256,8,39375824054,13881269278,0.352533,0.179002,2.484771,0.005770,50940066,8828286514


## Prefetch

x86 provide prefetch instructions. As a programmer, you may insert ```_mm_prefetch``` in x86 programs to perform software prefetch for your code. The gcc compiler also has a flag ```-fprefetch-loop-arrays``` to automatically insert software prefetch instructions.

### Using prefetch in matrix transpose code

The following example is a highly optimized matrix transpose code. In the example, we try to prefetch the next row.

In [18]:
render_code("./prefetch/transpose.cpp", lang="c++", show=["//START", "//END"])

// ./prefetch/transpose.cpp:42-63 (22 lines)
    //START
    //  Iterate each row.
    f_vector *y_iter = T;
    do{
        //  Iterate each column.
        f_vector *ptr_x = y_iter + block;
        f_vector *ptr_y = y_iter + row_size;

        do{

#ifdef ENABLE_PREFETCH
            _mm_prefetch((char*)(ptr_y + row_size),_MM_HINT_T0);
#endif
            swap_block(ptr_x,ptr_y,block);

            ptr_x += block;
            ptr_y += row_size;
        }while (ptr_y < stop_T);

        y_iter += iter_size;
    }while (y_iter < end);
    //END

Now, let's take a look of what's happening!

In [19]:
! cd prefetch; make clean; make
# ! echo "Without prefetch -- the baseline"; ssh htseng@celebi "lscpu | grep Model; cd courses/CS203/demo/memory/prefetch/; ./transpose"
! echo "Without prefetch -- the baseline"
! lscpu | grep Model
! ./prefetch/transpose
! echo "With prefetch"
! ./prefetch/transpose_prefetch

rm -f blockmm_sse blockmm blockmm_sse_prefetch transpose transpose_prefetch
g++ -msse4.1 -mavx -O3 transpose.cpp -o transpose 
g++ -msse4.1 -mavx -O3 -DENABLE_PREFETCH transpose.cpp -o transpose_prefetch 
Without prefetch -- the baseline
Model name:                           13th Gen Intel(R) Core(TM) i7-13700
Model:                                183
bytes = 4294967296
Starting Data Transpose...   Done
Time: 0.510648 seconds
With prefetch
bytes = 4294967296
Starting Data Transpose...   Done
Time: 0.430467 seconds


Let's try a different machine now.

In [20]:
! ssh htseng@xerneas "cd /nfshome/htseng/courses/CSE142/demo/software_optimizations_memory/; make -C ./prefetch clean; make -C ./prefetch ; lscpu | grep Model"
! echo "Without prefetch -- the baseline"; ssh htseng@xerneas  "/nfshome/htseng/courses/CSE142/demo/software_optimizations_memory/prefetch/transpose"
! echo "With prefetch";  ssh htseng@xerneas  "/nfshome/htseng/courses/CSE142/demo/software_optimizations_memory/prefetch/transpose_prefetch"

make: Entering directory '/nfshome/htseng/courses/CSE142/demo/memory/prefetch'
rm -f blockmm_sse blockmm blockmm_sse_prefetch transpose transpose_prefetch
make: Leaving directory '/nfshome/htseng/courses/CSE142/demo/memory/prefetch'
make: Entering directory '/nfshome/htseng/courses/CSE142/demo/memory/prefetch'
g++ -msse4.1 -mavx -O3 transpose.cpp -o transpose 
g++ -msse4.1 -mavx -O3 -DENABLE_PREFETCH transpose.cpp -o transpose_prefetch 
make: Leaving directory '/nfshome/htseng/courses/CSE142/demo/memory/prefetch'
Model name:                           AMD Ryzen 9 5950X 16-Core Processor
Model:                                33
Without prefetch -- the baseline
bytes = 1073741824
Starting Data Transpose...   Done
Time: 0.115764 seconds
With prefetch
bytes = 1073741824
Starting Data Transpose...   Done
Time: 0.108927 seconds


In [21]:
! ssh htseng@blissey "cd /nfshome/htseng/courses/CSE142/demo/memory/; make -C ./prefetch clean; make -C ./prefetch ; lscpu | grep Model"
! echo "Without prefetch -- the baseline"; ssh htseng@blissey  "/nfshome/htseng/courses/CSE142/demo/memory/prefetch/transpose"
! echo "With prefetch";  ssh htseng@blissey  "/nfshome/htseng/courses/CSE142/demo/memory/prefetch/transpose_prefetch"

make: Entering directory '/nfshome/htseng/courses/CSE142/demo/memory/prefetch'
rm -f blockmm_sse blockmm blockmm_sse_prefetch transpose transpose_prefetch
make: Leaving directory '/nfshome/htseng/courses/CSE142/demo/memory/prefetch'
make: Entering directory '/nfshome/htseng/courses/CSE142/demo/memory/prefetch'
g++ -msse4.1 -mavx -O3 transpose.cpp -o transpose 
g++ -msse4.1 -mavx -O3 -DENABLE_PREFETCH transpose.cpp -o transpose_prefetch 
make: Leaving directory '/nfshome/htseng/courses/CSE142/demo/memory/prefetch'
Model name:                           AMD Ryzen 7 5700X 8-Core Processor
Model:                                33
Without prefetch -- the baseline
bytes = 1073741824
Starting Data Transpose...   Done
Time: 0.103637 seconds
With prefetch
bytes = 1073741824
Starting Data Transpose...   Done
Time: 0.096679 seconds


In [22]:
! ssh htseng@eevee "cd /nfshome/htseng/courses/CSE142/demo/memory/; make -C ./prefetch clean; make -C ./prefetch ; lscpu | grep Model"
! echo "Without prefetch -- the baseline"; ssh htseng@eevee  "/nfshome/htseng/courses/CSE142/demo/memory/prefetch/transpose"
! echo "With prefetch";  ssh htseng@eevee  "/nfshome/htseng/courses/CSE142/demo/memory/prefetch/transpose_prefetch"

make: Entering directory '/nfshome/htseng/courses/CSE142/demo/memory/prefetch'
rm -f blockmm_sse blockmm blockmm_sse_prefetch transpose transpose_prefetch
make: Leaving directory '/nfshome/htseng/courses/CSE142/demo/memory/prefetch'
make: Entering directory '/nfshome/htseng/courses/CSE142/demo/memory/prefetch'
g++ -msse4.1 -mavx -O3 transpose.cpp -o transpose 
g++ -msse4.1 -mavx -O3 -DENABLE_PREFETCH transpose.cpp -o transpose_prefetch 
make: Leaving directory '/nfshome/htseng/courses/CSE142/demo/memory/prefetch'
Model:                              85
Model name:                         Intel(R) Xeon(R) Silver 4108 CPU @ 1.80GHz
Without prefetch -- the baseline
bytes = 1073741824
Starting Data Transpose...   Done
Time: 0.270896 seconds
With prefetch
bytes = 1073741824
Starting Data Transpose...   Done
Time: 0.238537 seconds



-- It doesn't work always!

In [23]:
render_code("matrix_mul/blockmm_interchange.c", show=["//START","//END"])

// matrix_mul/blockmm_interchange.c:59-77 (19 lines)
//START
void blockmm(double **a, double **b, double **c, uint64_t M, uint64_t N, uint64_t K)
{
  uint64_t i,j,k, ii, jj, kk;
  for(i = 0; i < M; i+=tile_size)
  {
    for(j = 0; j < K; j+=tile_size)
    {
      for(k = 0; k < N; k+=tile_size)
      {
          for(kk = k; kk < k+tile_size; kk++)
              for(ii = i; ii < i+tile_size; ii++)
                for(jj = j; jj < j+tile_size; jj++)              
                c[ii][jj] += a[ii][kk]*b[kk][jj];
      }
    }
  }  
}
//END

In [9]:
! cd matrix_mul; rm -f blockmm_interchange; make blockmm_interchange; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm_interchange.csv
! ./matrix_mul/blockmm_interchange 2048 8 >> ./matrix_mul/blockmm_interchange.csv 
! ./matrix_mul/blockmm_interchange 2048 16 >> ./matrix_mul/blockmm_interchange.csv 
! ./matrix_mul/blockmm_interchange 2048 32 >> ./matrix_mul/blockmm_interchange.csv 
! ./matrix_mul/blockmm_interchange 2048 64 >> ./matrix_mul/blockmm_interchange.csv
! ./matrix_mul/blockmm_interchange 2048 128 >> ./matrix_mul/blockmm_interchange.csv
! ./matrix_mul/blockmm_interchange 2048 256 >> ./matrix_mul/blockmm_interchange.csv
! cd matrix_mul; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm.csv
! ./matrix_mul/blockmm 2048 16 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 32 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 64 >> ./matrix_mul/blockmm.csv
! ./matrix_mul/blockmm 2048 128 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 256 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 512 >> ./matrix_mul/blockmm.csv 
display_df_mono(render_csv("matrix_mul/blockmm.csv"))
display_df_mono(render_csv("matrix_mul/blockmm_interchange.csv"))


gcc -O3 -DHAVE_LINUX_PERF_EVENT_H blockmm_interchange.c perfstats.c -o blockmm_interchange
blockmm_interchange.c: In function ‘main’:
blockmm_interchange.c:48:16: warning: format ‘%lu’ expects argument of type ‘long unsigned int’, but argument 3 has type ‘int’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wformat=-Wformat=]8;;]
   48 |   printf("%d,%lu,",ARRAY_SIZE,tile_size);
      |              ~~^              ~~~~~~~~~
      |                |              |
      |                |              int
      |                long unsigned int
      |              %u
10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,10521102336.000000,

,index,size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,2048,16,74473386911,19237819138,0.258318,0.193622,3.724860,0.071290,2573932385,36105251964
1,1,2048,32,71544344489,28028089435,0.391758,0.193641,5.427381,0.217403,7655790077,35214730065
2,2,2048,64,70028007243,32675835340,0.466611,0.194037,6.340316,0.242686,8429345312,34733602612
3,3,2048,128,69474313293,37149820875,0.534727,0.193606,7.192432,0.244877,8470773775,34591994955
4,4,2048,256,69120040189,53298747450,0.771104,0.193899,10.334574,0.241203,8316237916,34478129179
5,5,2048,512,69044315168,72573739373,1.051118,0.193706,14.057958,0.235176,8106169032,34468477710


,index,size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,2048,8,68604448245,15718358160,0.229116,0.193676,3.044266,0.013131,316092777,24071954131
1,1,2048,16,50678840944,13624971615,0.268849,0.193750,2.639845,0.072926,1318845843,18084631718
2,2,2048,32,42341050047,10245513297,0.241976,0.193869,1.986286,0.075526,1162781546,15395852107
3,3,2048,64,38315079204,8809988526,0.229935,0.193705,1.706541,0.076670,1082713591,14121753798
4,4,2048,128,36334188565,8223440647,0.226328,0.193631,1.592312,0.076101,1027416171,13500753876
5,5,2048,256,35355990869,9389590026,0.265573,0.193904,1.820680,0.030850,407096195,13195857929
